# Exploración de variables

Análisis bivariado con el 80 % de entrenamiento. No entrena modelos.

In [1]:
import pandas as pd
from IPython.display import display

from bolsalab.explain import GLOBAL_FILE, INTERACTIONS_FILE
from bolsalab.exploration import (
    QUESTIONS, VARIABLES, cross, load_training_people, sell_rate, summary, surprising_cells,
)

people = load_training_people()


def porcentajes(tabla):
    """Muestra una tabla con un decimal y las casillas más altas más oscuras."""
    return tabla.style.format(precision=1, na_rep="").background_gradient(cmap="Oranges", axis=None)

C:\Repos\bolsalab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Cada variable sola

**1 · Tabla.** Personas y venta general

In [2]:
pd.DataFrame({"personas": [len(people)], "vendió (%)": [people["sold"].mean() * 100]}).round(1)

,personas,vendió (%)
0,43763,19.9


**2 · Tabla.** Venta por respuesta: una variable fuerte y una débil

In [3]:
display(sell_rate(people, "male").round(1))
display(sell_rate(people, "married").round(1))

,personas,vendió (%)
male,,
0,14415,12.2
1,29348,23.6


,personas,vendió (%)
married,,
0,13359,20.4
1,30404,19.6


**3 · Tabla.** Venta por décadas de edad

In [4]:
sell_rate(people, "age_group").round(1)

,personas,vendió (%)
age_group,,
<20,33,30.3
20-29,3143,19.3
30-39,10273,18.0
40-49,12202,18.2
50-59,10433,19.6
60-69,5721,25.1
70-79,1850,27.5
80+,108,17.6


**4 · Tabla.** Resumen de las 16 variables, ordenado por diferencia (grupos de 500 personas o más)

In [5]:
resumen = summary(people)
resumen.style.format(precision=1).format({"IV": "{:.3f}"}).bar(subset=["diferencia"], color="#f4b183")

,variable,vende más (%),grupo que más vende,vende menos (%),grupo que menos vende,diferencia,IV
0,risk_aversion,31.872260,0.900000,16.199058,0.400000,15.673203,0.041
1,income_bracket,31.783920,1,17.680196,6,14.103723,0.029
2,short_term_view,31.291391,5,17.256684,2,14.034706,0.027
3,male,23.616601,1,12.237253,0,11.379348,0.126
4,knows_diversification,27.721638,0,17.661866,1,10.059771,0.063
5,knowledge_confidence,27.069767,5,17.025175,2,10.044592,0.025
6,age_group,27.513514,70-79,18.008371,30-39,9.505142,0.028
7,overconfidence,28.424554,1,19.168974,0,9.255580,0.021
8,employed,26.762549,0,19.260512,1,7.502038,0.015
9,assets_bracket,24.965986,1,17.602571,5,7.363415,0.021


**5 · Tabla.** Las mismas variables ordenadas por valor de información (IV)

In [6]:
por_iv = resumen[["variable", "IV"]].sort_values("IV", ascending=False, ignore_index=True)
por_iv["lectura"] = pd.cut(por_iv["IV"], [0, 0.02, 0.1, 0.3, 10], right=False,
                           labels=["no sirve sola", "débil", "media", "fuerte"])
por_iv.style.format({"IV": "{:.3f}"}).bar(subset=["IV"], color="#f4b183")

,variable,IV,lectura
0,male,0.126,media
1,knows_diversification,0.063,débil
2,risk_aversion,0.041,débil
3,income_bracket,0.029,débil
4,age_group,0.028,débil
5,short_term_view,0.027,débil
6,knows_interest,0.025,débil
7,knowledge_confidence,0.025,débil
8,knows_inflation,0.024,débil
9,assets_bracket,0.021,débil


## Variables nuevas

**6 · Tabla.** Total de aciertos: sumar las tres preguntas de finanzas

In [7]:
people["correct_answers"] = people[QUESTIONS].sum(axis=1)
sell_rate(people, "correct_answers").round(1)

,personas,vendió (%)
correct_answers,,
0,2370,31.9
1,4350,27.3
2,10154,21.2
3,26889,17.1


**7 · Tabla.** Exceso de confianza: la columna actual frente a la definición del artículo

In [8]:
pocos_aciertos = people["correct_answers"] < people["correct_answers"].mean()
confiado = people["knowledge_confidence"] >= 4
people["overconfidence_article"] = (pocos_aciertos & confiado).astype(int)

display(pd.crosstab(people["overconfidence"], people["overconfidence_article"],
                    rownames=["columna actual"], colnames=["definición del artículo"]))
display(sell_rate(people, "overconfidence").round(1))
display(sell_rate(people, "overconfidence_article").round(1))

definición del artículo,0,1
columna actual,,
0,40456,0
1,800,2507


,personas,vendió (%)
overconfidence,,
0,40456,19.2
1,3307,28.4


,personas,vendió (%)
overconfidence_article,,
0,41256,19.3
1,2507,30.0


## Cruces de dos variables

**8 · Tabla.** Confianza (filas) por total de aciertos (columnas): % que vendió

In [9]:
porcentajes(cross(people, "knowledge_confidence", "correct_answers"))

correct_answers,0,1,2,3
knowledge_confidence,,,,
1,24.2,24.6,16.0,13.3
2,26.9,22.9,18.0,14.5
3,37.8,30.4,23.7,17.6
4,,,26.0,19.8
5,,,,23.4


**9 · Tabla.** Hombre (filas) por total de aciertos (columnas): % que vendió

In [10]:
porcentajes(cross(people, "male", "correct_answers"))

correct_answers,0,1,2,3
male,,,,
0,21.5,17.2,12.4,9.2
1,42.1,37.1,26.6,20.0


**10 · Tabla.** Edad (filas) por si trabaja (columnas): % que vendió

In [11]:
porcentajes(cross(people, "age_group", "employed"))

employed,0,1
age_group,,
<20,,
20-29,,19.4
30-39,,17.8
40-49,,18.2
50-59,,19.4
60-69,27.4,24.3
70-79,29.1,24.8
80+,,


**11 · Tabla.** Las 15 combinaciones que más se alejan de lo esperado

In [12]:
ya_combinadas = [*QUESTIONS, "overconfidence"]
candidatas = [v for v in VARIABLES if v not in ya_combinadas] + ["correct_answers"]
surprising_cells(people, candidatas).head(15).round(1)

,variable 1,respuesta 1,variable 2,respuesta 2,personas,vendió (%),esperado (%),sorpresa
0,knowledge_confidence,1,short_term_view,1,1045,27.3,19.0,8.3
1,short_term_view,1,correct_answers,1,507,36.5,29.1,7.4
2,male,1,correct_answers,0,1190,42.1,35.6,6.5
3,short_term_view,4,correct_answers,1,690,20.9,27.2,-6.3
4,male,1,correct_answers,1,2219,37.1,31.1,6.0
5,age_group,70-79,employed,0,1173,29.1,34.4,-5.3
6,income_bracket,3,correct_answers,0,518,29.5,34.8,-5.2
7,knowledge_confidence,1,correct_answers,0,612,24.2,29.3,-5.1
8,risk_aversion,0.9,correct_answers,3,847,24.1,29.1,-5.0
9,assets_bracket,3,short_term_view,1,663,15.8,20.8,-5.0


## Lo que usa el modelo oficial (SHAP)

**12 · Tabla.** Peso de cada variable en el modelo, junto a su IV

In [13]:
peso = pd.read_csv(GLOBAL_FILE)[["variable", "peso_medio"]].replace("age", "age_group")
junto = resumen[["variable", "IV"]].merge(peso).sort_values("peso_medio", ascending=False, ignore_index=True)
junto.style.format(precision=3).bar(subset=["peso_medio", "IV"], color="#f4b183")

,variable,IV,peso_medio
0,male,0.126,0.327
1,knows_diversification,0.063,0.170
2,knowledge_confidence,0.025,0.109
3,knows_inflation,0.024,0.083
4,age_group,0.028,0.082
5,risk_aversion,0.041,0.077
6,short_term_view,0.027,0.076
7,income_bracket,0.029,0.062
8,knows_interest,0.025,0.059
9,assets_bracket,0.021,0.058


**13 · Tabla.** Los 10 pares de variables que el modelo más usa juntos

In [14]:
pares = pd.read_csv(INTERACTIONS_FILE)
pares.insert(0, "puesto", pares.index + 1)
pares.head(10).round(4)

,puesto,variable 1,variable 2,fuerza
0,1,knows_diversification,age,0.0306
1,2,knows_inflation,knows_diversification,0.0283
2,3,knowledge_confidence,male,0.0273
3,4,knows_diversification,male,0.0262
4,5,age,risk_aversion,0.0225
5,6,knows_inflation,risk_aversion,0.0221
6,7,risk_aversion,short_term_view,0.0211
7,8,knowledge_confidence,short_term_view,0.0205
8,9,male,university,0.0165
9,10,knowledge_confidence,age,0.0163


**14 · Tabla.** Puesto de los pares formados con los ingredientes de las candidatas (de 120 pares)

In [15]:
ingredientes = [*QUESTIONS, "knowledge_confidence", "male", "income_bracket", "assets_bracket"]
entre_ingredientes = pares["variable 1"].isin(ingredientes) & pares["variable 2"].isin(ingredientes)
pares[entre_ingredientes].head(12).round(4)

,puesto,variable 1,variable 2,fuerza
1,2,knows_inflation,knows_diversification,0.0283
2,3,knowledge_confidence,male,0.0273
3,4,knows_diversification,male,0.0262
10,11,male,income_bracket,0.0155
11,12,knows_inflation,knowledge_confidence,0.0153
13,14,knows_diversification,income_bracket,0.0141
14,15,knows_interest,knows_diversification,0.0139
20,21,knows_inflation,male,0.0112
22,23,knowledge_confidence,assets_bracket,0.0106
24,25,knows_interest,male,0.0085


## Cruce de la cuarta candidata

**15 · Tabla.** Ingreso (filas) por activos (columnas), en tres niveles: % que vendió

In [16]:
people["income_level"] = pd.cut(people["income_bracket"], [0, 2, 8, 12],
                                labels=["bajo (1-2)", "medio (3-8)", "alto (9-12)"])
people["assets_level"] = pd.cut(people["assets_bracket"], [0, 2, 7, 10],
                                labels=["bajo (1-2)", "medio (3-7)", "alto (8-10)"])
porcentajes(cross(people, "income_level", "assets_level"))

assets_level,bajo (1-2),medio (3-7),alto (8-10)
income_level,,,
bajo (1-2),35.0,25.7,
medio (3-8),20.5,18.0,21.4
alto (9-12),,18.9,20.7
